In [1]:
model = "llama3.2:1b"

#### Task 1: Create a Simple Chain for Summarization

**Objective:**

Build a LangChain chain that can summarize a given text.

**Task Description:**

- Create a llm chain using with a Ollama model.
- Define a prompt template for summarization. The summary should be only one sentence.
- Run the chain with a sample text and print the summary.
- Add model output streaming.
- Run the chain with streaming with a sample text and print the summary.

**Hint: The five messages in LangChain:**

- SystemMessage: corresponds to system role
- HumanMessage: corresponds to user role
- AIMessage: corresponds to assistant role
- AIMessageChunk: corresponds to assistant role, used for streaming responses
- ToolMessage: corresponds to tool role

[More Information](https://python.langchain.com/docs/concepts/messages/)

**Useful links:**

- [How To Prompt Template 1](https://python.langchain.com/v0.2/docs/tutorials/extraction/#the-extractor)
- [How To Prompt Template 2](https://python.langchain.com/v0.2/api_reference/core/prompts/langchain_core.prompts.chat.ChatPromptTemplate.html#langchain_core.prompts.chat.ChatPromptTemplate)
- [How To LCEL Chains 1](https://python.langchain.com/v0.2/docs/concepts/#langchain-expression-language-lcel)
- [How To LCEL Chains 2](https://python.langchain.com/v0.2/docs/versions/migrating_chains/llm_chain/#lcel)
- [How To Chain Streaming](https://python.langchain.com/v0.2/docs/concepts/#streaming)


In [2]:
from langchain_ollama.chat_models import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser


# Load the Ollama model
# HINWEIS: model variable muss in der Zelle davor gesetzt und das Modell geladen sein!
llm = ChatOllama(model=model)

# Define the prompt template
summarization_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an expert summarizer. Summarize the following text into exactly one short, concise sentence."),
    ("human", "{text}"),
])

# Create the LLMChain (LCEL: Prompt | LLM | OutputParser)
summarization_chain = summarization_prompt | llm | StrOutputParser()


In [3]:
text="hallo"
# Stream the chain output
for chunk in summarization_chain.stream({"text": text}):
    print(chunk,  end="", flush=True)

It seems you haven't provided any text for me to summarize. Please go ahead and share the text, and I'll be happy to assist!

#### Task 2: Chain with Tool Usage (Simple Math Tool)

**Objective:**

Create a LangChain chain that uses a simple math tool to perform calculations.

**Task Description:**

- Define a function as tool which multiplies two integer values and return the result.
- Create a chain
- Print the result of the calculation.

**Useful links:**

- [How To Tools](https://python.langchain.com/v0.2/docs/how_to/tools_chain/#create-a-tool)
- [How To Tools in Chains](https://python.langchain.com/v0.2/docs/how_to/tools_chain/#chains)
- [How To Tool Calling](https://python.langchain.com/v0.2/docs/concepts/#functiontool-calling)
- [How To Chain and Call Tools with Ollama](https://python.langchain.com/v0.2/docs/integrations/chat/ollama/)


In [4]:
from langchain_core.tools import tool

# ADD HERE YOUR CODE
# Create custom tool
@tool
def multiply(first_int: int, second_int: int) -> int:
    """Multiplies two integer values and returns the result."""
    return first_int * second_int


print(multiply.name)
print(multiply.description)
print(multiply.args) # -> definition of tool arguments

# Invoke custom tool
multiply.invoke({"first_int": 4, "second_int": 5})

multiply
Multiplies two integer values and returns the result.
{'first_int': {'title': 'First Int', 'type': 'integer'}, 'second_int': {'title': 'Second Int', 'type': 'integer'}}


20

In [5]:
# Load the Ollama model
llm = ChatOllama(model=model)

# ADD HERE YOUR CODE
# Use bind_tools to pass the definition of our tool in as part of each call to the model, so that the model can invoke the tool
llm_with_tools = llm.bind_tools([multiply])

# When the model invokes the tool, this will show up in the AIMessage.tool_calls attribute of the output -> extract tool parameters from input text
msg = llm_with_tools.invoke("whats 5 times forty two")
print(msg.tool_calls)

[{'name': 'multiply', 'args': {'first_int': '42', 'second_int': '5'}, 'id': '41f35385-31fd-4417-a0ba-a362e245444a', 'type': 'tool_call'}]


In [7]:
# ADD HERE YOUR CODE
# Create the chain: pass the extracte tool parameters from the input text to the tool -> extract the arguments of the first tool_call
chain_with_tools = llm_with_tools | (lambda x: x.tool_calls[0]["args"]) | multiply 
#x ist AiMessage, das greift auf tool calls zu (liste von Tool Aufrufen), [0] = erster Tool aus liste

# Run chain
chain_with_tools.invoke("whats 5 times forty two")

210

#### Task 3: Agent with Tool Usage (Two Tools)

**Objective:**

Create a LangChain agent that uses two tools to perform tasks.

**Task Description:**

- Define prompt template.
- Define tools.
- Create an Agent using the Ollama model, prompt template and tools.
- Run the agent with a prompt that requires one or both tools.
- Observe how the agent uses the tools to complete the task.

**Useful links:**

- [How To 1](https://python.langchain.com/v0.2/docs/concepts/#agents)
- [How To 2](https://python.langchain.com/v0.2/docs/how_to/tools_chain/#agents)
- [How To 3](https://python.langchain.com/v0.2/docs/tutorials/agents/)


In [9]:
from langchain.agents import AgentExecutor, create_tool_calling_agent
from langchain_ollama.chat_models import ChatOllama
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.tools import tool
# Import StrOutputParser, falls nötig (für Task 1 verwendet, aber hier nicht zwingend)
# from langchain_core.output_parsers import StrOutputParser

# Lade das Ollama-Modell
# Stelle sicher, dass `model` (z.B. "llama3.2:1b") in Zelle 1 gesetzt ist
llm = ChatOllama(model=model)

# Custom math tools
@tool
def add(first_int: int, second_int: int) -> int:
    """Add two integers."""
    return first_int + second_int


@tool
def exponentiate(base: int, exponent: int) -> int:
    """Exponentiate the base to the exponent power."""
    return base**exponent


tools = [add, exponentiate]

# Definiere das Prompt Template
# WICHTIG: Agenten-Prompts benötigen einen MessagesPlaceholder für das 'agent_scratchpad'
agent_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful math agent. You have access to tools for adding and exponentiating. Use them to answer the user's questions."),
    ("human", "{input}"), # Verwende {input} als Variable für die Benutzerfrage
    MessagesPlaceholder(variable_name="agent_scratchpad"), # WICHTIG: Platzhalter für die Agenten-Schritte
])

In [10]:
# Construct the tool calling agent
agent_with_tools = create_tool_calling_agent(
    llm=llm,
    tools=tools,
    prompt=agent_prompt,
)

# Create an agent executor by passing in the agent and tools
agent_executor_with_tools = AgentExecutor(
    agent=agent_with_tools,
    tools=tools,
    verbose=True, # Optional: zeigt die Schritte des Agenten an
)

In [11]:
# Run chain
agent_executor_with_tools.invoke(
    {
        "input": "First take 3 to the power of five and afterwards add 12."
    }
)



> Entering new AgentExecutor chain...

Invoking: `add` with `{'first_int': 5, 'second_int': 12}`


17The result of 3 to the power of 5 is 243, and adding 12 gives us 255.

> Finished chain.


{'input': 'First take 3 to the power of five and afterwards add 12.',
 'output': 'The result of 3 to the power of 5 is 243, and adding 12 gives us 255.'}

#### [Optional] Task 4: Enhance Agent with Memory

**Objective:**

Eenhance the agent from Task 3 with memory to improve its context awareness and ability to maintain state.

**Instructions:**

- Create a ConversationBufferMemory to store chat history.
- Modify the agent to use the memory to inform its responses.
- Run the agent with a series of prompts that require context or state to be maintained.
- Observe how the agent's responses improve with the addition of memory.

**Useful links:**

- [How To Memory 1](https://python.langchain.com/v0.2/api_reference/langchain/memory/langchain.memory.buffer.ConversationBufferMemory.html#langchain.memory.buffer.ConversationBufferMemory)
- [How To Memory 1](https://python.langchain.com/v0.2/docs/versions/migrating_chains/conversation_chain/#legacy)


In [12]:
from langchain.agents import AgentExecutor, create_tool_calling_agent
from langchain.memory import ConversationBufferMemory
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_ollama.chat_models import ChatOllama

# Lade das Ollama-Modell
# Stelle sicher, dass `model` (z.B. "llama3.2:1b") in Zelle 1 gesetzt ist
llm = ChatOllama(model=model)

# Definiere Memory-Objekt für Konversationshistorie
# ACHTUNG: Der memory_key muss als MessagesPlaceholder im Prompt verwendet werden!
memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True, output_key="output")

# Custom math tools (erneut definieren oder aus Zelle 3 importieren/kopieren)
# Damit diese Zelle auch alleine funktioniert, werden die Tools hier wiederholt
@tool
def add(first_int: int, second_int: int) -> int:
    """Add two integers."""
    return first_int + second_int

@tool
def exponentiate(base: int, exponent: int) -> int:
    """Exponentiate the base to the exponent power."""
    return base**exponent

tools = [add, exponentiate]

# Füge History-Platzhalter zum Prompt hinzu
agent_prompt_with_memory = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful math agent with memory. Use the tools to answer the user's questions. Maintain context from previous messages."),
    MessagesPlaceholder(variable_name="chat_history"), # Platzhalter für den Verlauf
    ("human", "{input}"), # Verwende {input}
    MessagesPlaceholder(variable_name="agent_scratchpad"), # WICHTIG: Platzhalter für die Agenten-Schritte
])

# Construct the tool calling agent
agent_with_tools_and_memory = create_tool_calling_agent(
    llm=llm,
    tools=tools,
    prompt=agent_prompt_with_memory,
)

# Create an agent executor by passing in the agent and tools
agent_executor_with_tools_and_memory = AgentExecutor(
    agent=agent_with_tools_and_memory,
    tools=tools,
    memory=memory, # Speicher dem Executor übergeben
    verbose=True,
    handle_parsing_errors=True,
    # HINWEIS: Bei Verwendung von ConversationBufferMemory wird der 'input' Key erwartet, der zu {input} im Prompt passt.
)

In [13]:
# Erste Frage: Berechne etwas
question = "Take 3 to the fifth power then add that 12?"
ai_msg_1 = agent_executor_with_tools_and_memory.invoke({"input": question}) # Verwende 'input'
print(f"Erste Antwort: {ai_msg_1['output']}\n")

# Zweite Frage: Erkläre basierend auf der vorherigen Berechnung
second_question = "Explain how you have calculated the result."
ai_msg_2 = agent_executor_with_tools_and_memory.invoke({"input": second_question}) # Verwende 'input'
print(f"Zweite Antwort: {ai_msg_2['output']}")

# Optional: Überprüfe den Inhalt des Speichers
# print("\nMemory Content:")
# print(memory.load_memory_variables({}))



> Entering new AgentExecutor chain...

Invoking: `add` with `{'first_int': '3', 'properties': {'base': '12'}, 'second_int': '5'}`


8The result of taking 3 to the fifth power and adding 12 is:

3^5 = 243
243 + 12 = 255

> Finished chain.
Erste Antwort: The result of taking 3 to the fifth power and adding 12 is:

3^5 = 243
243 + 12 = 255



> Entering new AgentExecutor chain...

Invoking: `exponentiate` with `{'base': '3', 'exponent': '5'}`


243The calculation of 3 to the fifth power is done by raising 3 to the power of 5, which means multiplying 3 by itself 5 times.

3 × 3 = 9
9 × 3 = 27
27 × 3 = 81
81 × 3 = 243

So, 3^5 equals 243. 

Now, let's add 12 to that result:

243 + 12 = 255

> Finished chain.
Zweite Antwort: The calculation of 3 to the fifth power is done by raising 3 to the power of 5, which means multiplying 3 by itself 5 times.

3 × 3 = 9
9 × 3 = 27
27 × 3 = 81
81 × 3 = 243

So, 3^5 equals 243. 

Now, let's add 12 to that result:

243 + 12 = 255
